# Merge syst dfs from grid job

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot
from matplotlib import gridspec

import sys
sys.path.append('../../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
import pyanalib.stat_helpers as sh
from makedf.util import *
from analysis_village.plot_style.plot_helper import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
sample_str = "v10_14_02_02+"

## Open files and concat

In [ ]:
#input_path = "/data/sungbino/sbnd/gen2/cohpi/syst_no_dirz_10k/"
input_path = "/data/sungbino/sbnd/gen2/cohpi/cohpi_all_weight_signal_mc_no_dirz_cont_var_10k/"
#prefix = "cohpi_all_weight_signal_mc_no_dirz_"
prefix = "cohpi_all_weight_signal_mc_no_dirz_cont_var_10k_"

### Test 1 file

In [ ]:
print_keys(input_path + prefix + "103.df")

In [ ]:
keys2load = ['hdr', "mcnu", 'evt', "pot", "histpotdf", "histgenevtdf"]

In [ ]:
test_dfs = load_dfs(input_path + prefix + "103.df", keys2load, n_max_concat=1)

In [ ]:
test_dfs['mcnu']

In [ ]:
test_dfs['evt'].index = test_dfs['evt'].index.droplevel(1)

# 2. Move 'entry' and 'rec.slc..index' into the index (appending to '__ntuple')
test_dfs['evt'] = test_dfs['evt'].set_index(['entry', 'rec.slc..index'], append=True)

In [ ]:
test_dfs['evt']

### Concat all

In [ ]:
from tqdm import tqdm
import pandas as pd
import warnings
#warnings.simplefilter("ignore", category=FutureWarning)

# 1. Initialize empty lists to hold the dataframes and indices
evt_list = []
mcnu_list = []
pot_list = []
hdr_list = []
histpotdf_list = []
histgenevtdf_list = []
file_indices = [] # NEW: Keep track of 'i' for the concatenation

for i in tqdm(range(0, 10000), desc="Loading DataFrames"):
    # Load data
    test_dfs = load_dfs(input_path + prefix + f"{i}.df", keys2load, n_max_concat=1)

    # Process the evt DataFrame
    this_evt_df = test_dfs['evt']
    this_evt_df.index = this_evt_df.index.droplevel(1)
    this_evt_df = this_evt_df.set_index(['entry', 'rec.slc..index'], append=True)

    # 2. Append to lists INSTEAD of concatenating in the loop
    evt_list.append(this_evt_df)
    mcnu_list.append(test_dfs['mcnu'])
    pot_list.append(test_dfs['pot'])
    hdr_list.append(test_dfs['hdr'])
    histpotdf_list.append(test_dfs['histpotdf'])
    histgenevtdf_list.append(test_dfs['histgenevtdf'])
    
    file_indices.append(i) # NEW: Store the loop index

# 3. Concatenate everything ONCE at the end using the 'keys' argument.
# This automatically prepends 'i' as an outermost index level called 'file_idx'.
evt_df = pd.concat(evt_list, keys=file_indices, names=['file_idx']) 
mcnu_df = pd.concat(mcnu_list, keys=file_indices, names=['file_idx']) 
pot_df = pd.concat(pot_list, keys=file_indices, names=['file_idx'])
hdr_df = pd.concat(hdr_list, keys=file_indices, names=['file_idx'])
histpotdf_df = pd.concat(histpotdf_list, keys=file_indices, names=['file_idx'])
histgenevtdf_df = pd.concat(histgenevtdf_list, keys=file_indices, names=['file_idx'])

In [ ]:
evt_df

In [ ]:
mcnu_df

In [ ]:
hdr_df

In [ ]:
histpotdf_df

In [ ]:
histgenevtdf_df

In [ ]:
hdr_df.pot.sum()

In [ ]:
histpotdf_df.TotalPOT.sum()

In [ ]:
histgenevtdf_df.TotalGenEvents.sum()

### Save the df

In [ ]:
with pd.HDFStore('cohpi_aurora_5p4e20_systs_no_dirz.df') as store:
    store.put('evt', evt_df, format='fixed')
    store.put('mcnu', mcnu_df, format='fixed')
    store.put('pot', pot_df, format='fixed')
    store.put('hdr', hdr_df, format='fixed')
    store.put('histpotdf', histpotdf_df, format='fixed')
    store.put('histgenevtdf', histgenevtdf_df, format='fixed')